# Match Results Predictor

> In this Jupiter Notebook, I will used Advanced AI models in order to predict football match results.


### 1 Data Loading and Preperation

In [ ]:
import pandas as pd
import glob
import os

# Columns to keep

KEEP_COLS = ["Div", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR", "Attendance", "Referee", "HS", "AS", "HST", "AST", "HHW", "AHW", "HC", "AC", "HF", "AF", "HO", "AO", "HY", "AY", "HR", "AR", "HBP", "ABP"]

# Load and combine all CSVs 

CSV_DIR = "../Data/Premier League"

all_files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
print(f"Found {len(all_files)} CSV files")

dfs = []
for path in all_files:
    season = os.path.basename(path).replace(".csv", "")
    for encoding in ["utf-8-sig", "latin-1", "windows-1252"]:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                on_bad_lines="skip",
                engine="python",
            )
            break
        except UnicodeDecodeError:
            continue
    df["Season"] = season
    dfs.append(df)

print(f"\nLoaded {len(dfs)} files successfully")

raw = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(raw)}")

# Keep only relevant columns, filling missing ones with NaN

data = raw.reindex(columns=KEEP_COLS + ["Season"])

# Basic cleanup

def parse_date(val):
    for fmt in ["%d/%m/%Y", "%d/%m/%y"]:
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

data["Date"] = data["Date"].apply(parse_date)

data = data.dropna(subset=["FTR"])

data = data.sort_values("Date").reset_index(drop=True)

# Summary

print(f"\nMissing values per column:")
missing = data.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("  None!")
else:
    for col, count in missing.items():
        pct = count / len(data) * 100
        print(f"  {col:<10} {count:>5} missing  ({pct:.1f}%)")

Found 26 CSV files

Loaded 26 files successfully
Total rows: 9720

Missing values per column:
  Attendance  8960 missing  (92.2%)
  HHW         8959 missing  (92.2%)
  AHW         8959 missing  (92.2%)
  HO          8959 missing  (92.2%)
  AO          8959 missing  (92.2%)
  HBP         8959 missing  (92.2%)
  ABP         8959 missing  (92.2%)


### 2 Adding Gx Columns

The meaning of xG in football is “expected goals.” It’s a statistic used to measure how likely a shot is to become a goal.

Each shot gets a value between 0 and 1 based on factors like:

distance from goal
angle
whether it was a header or a shot with the foot
whether it was a counterattack
how many defenders were nearby
the type of pass before the shot

In [ ]:
import asyncio
import aiohttp
import understat
import pandas as pd
from tqdm import tqdm

# Fetch xG data from Understat
# Understat covers Premier League from 2014/15 onwards
# Seasons before that will have NaN for xG columns

SEASONS = list(range(2014, 2026))

async def fetch_season(session, year):
    u = understat.Understat(session)
    matches = await u.get_league_results("EPL", year)
    rows = []
    for m in matches:
        rows.append({
            "Date"     : m["datetime"][:10],
            "HomeTeam" : m["h"]["title"],
            "AwayTeam" : m["a"]["title"],
            "xG_home"  : float(m["xG"]["h"]),
            "xG_away"  : float(m["xG"]["a"]),
        })
    return rows

async def fetch_all():
    all_rows = []
    async with aiohttp.ClientSession() as session:
        for year in tqdm(SEASONS, desc="Fetching xG"):
            try:
                rows = await fetch_season(session, year)
                all_rows.extend(rows)
                print(f"  {year}/{str(year+1)[-2:]}  →  {len(rows)} matches")
            except Exception as e:
                print(f"  {year}  →  FAILED: {e}")
    return all_rows

# Running the async fetcher (works in Jupyter with nest_asyncio)
import nest_asyncio
nest_asyncio.apply()

rows = asyncio.run(fetch_all())
xg_df = pd.DataFrame(rows)
xg_df["Date"] = pd.to_datetime(xg_df["Date"])

print(f"\nFetched {len(xg_df)} xG records")

# Normalise team names to match football-data.co.uk because understat uses different team names

NAME_MAP = {
    "Manchester United"  : "Man United",
    "Manchester City"    : "Man City",
    "Newcastle United"   : "Newcastle",
    "Tottenham"          : "Tottenham",
    "Wolverhampton"      : "Wolves",
    "Sheffield United"   : "Sheffield United",
    "West Bromwich"      : "West Brom",
    "Nottingham Forest"  : "Nott'm Forest",
    "Leeds"              : "Leeds",
    "Leicester"          : "Leicester",
}

xg_df["HomeTeam"] = xg_df["HomeTeam"].replace(NAME_MAP)
xg_df["AwayTeam"] = xg_df["AwayTeam"].replace(NAME_MAP)

# Merge into main dataset

data = pd.merge(
    data,
    xg_df,
    on=["Date", "HomeTeam", "AwayTeam"],
    how="left",   # keep all rows, NaN for seasons before 2014/15
)

# Summary

total     = len(data)
with_xg   = data["xG_home"].notna().sum()
without   = total - with_xg

print(f"\n{'─'*45}")
print(f"Total matches : {total:,}")
print(f"With xG       : {with_xg:,}  ({with_xg/total*100:.1f}%)")
print(f"Without xG    : {without:,}")

data[["Date", "HomeTeam", "AwayTeam", "FTR", "xG_home", "xG_away"]].tail(10)
data.to_csv("../Data/combined_data.csv", index=False)

Fetching xG:   8%|▊         | 1/12 [00:01<00:13,  1.19s/it]

  2014/15  →  380 matches


Fetching xG:  17%|█▋        | 2/12 [00:01<00:08,  1.18it/s]

  2015/16  →  380 matches


Fetching xG:  25%|██▌       | 3/12 [00:02<00:06,  1.37it/s]

...

  2023/24  →  380 matches


Fetching xG:  92%|█████████▏| 11/12 [00:05<00:00,  2.72it/s]

  2024/25  →  380 matches


Fetching xG: 100%|██████████| 12/12 [00:05<00:00,  2.23it/s]

  2025/26  →  349 matches

Fetched 4529 xG records

─────────────────────────────────────────────
Total matches : 9,719
With xG       : 3,945  (40.6%)
Without xG    : 5,774


### 3 Adding Sentiment Columns using an LLM

In [3]:
from transformers import pipeline

print("Loading sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    truncation=True,
    max_length=512
)
print("Model loaded!")

c:\Users\angel\Files\UCLL\Semester 4\Advanced AI\Match Results Project\VSC - Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30012.64it/s]


Model loaded!


In [ ]:
import requests
import pandas as pd
import time
from datetime import timedelta

GUARDIAN_API_KEY = "8289721f-2ef8-4cd3-b2a3-bd48ba877685"

data["Date"] = pd.to_datetime(data["Date"])
recent = data[data["Date"].dt.year >= 2020].copy().reset_index(drop=True)

# Load checkpoint if exists (so you can resume if interrupted)
import os
CHECKPOINT = "../Data/sentiment_checkpoint.csv"
if os.path.exists(CHECKPOINT):
    checkpoint_df = pd.read_csv(CHECKPOINT)
    checkpoint_df["Date"] = pd.to_datetime(checkpoint_df["Date"])
    done_indices = set(checkpoint_df.index)
    print(f"Resuming from checkpoint — {len(done_indices)} already done")
else:
    checkpoint_df = recent.copy()
    checkpoint_df["home_sentiment"] = None
    checkpoint_df["away_sentiment"] = None
    done_indices = set()
    print("Starting fresh")

# Fetch headlines from Guardian API

def fetch_guardian_headlines(team, match_date, days_before=5):
    date_from = (match_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
    date_to   = match_date.strftime("%Y-%m-%d")
    try:
        r = requests.get(
            "https://content.guardianapis.com/search",
            params={
                "q"           : f"{team} football",
                "section"     : "football",
                "from-date"   : date_from,
                "to-date"     : date_to,
                "page-size"   : 5,
                "order-by"    : "relevance",
                "show-fields" : "headline,trailText",
                "api-key"     : GUARDIAN_API_KEY,
            },
            timeout=10
        )
        results = r.json().get("response", {}).get("results", [])
        texts = []
        for a in results:
            fields = a.get("fields", {})
            headline = fields.get("headline", a.get("webTitle", ""))
            trail    = fields.get("trailText", "")
            texts.append(f"{headline}. {trail}".strip())
        return texts if texts else None
    except Exception:
        return None

# Score sentiment with HuggingFace model

LABEL_SCORE = {"LABEL_0": -1.0, "LABEL_1": 0.0, "LABEL_2": 1.0}

def score_headlines(texts):
    if not texts:
        return None
    try:
        results = sentiment_model(texts)
        scores  = [LABEL_SCORE[r["label"]] * r["score"] for r in results]
        return round(sum(scores) / len(scores), 4)
    except Exception:
        return None

# Process all matches

for i, row in checkpoint_df.iterrows():
    if pd.notna(checkpoint_df.at[i, "home_sentiment"]):
        continue  # already processed, skip

    home_texts = fetch_guardian_headlines(row["HomeTeam"], row["Date"])
    away_texts = fetch_guardian_headlines(row["AwayTeam"], row["Date"])

    checkpoint_df.at[i, "home_sentiment"] = score_headlines(home_texts)
    checkpoint_df.at[i, "away_sentiment"] = score_headlines(away_texts)

    # Save checkpoint every 50 matches
    if i % 50 == 0:
        checkpoint_df.to_csv(CHECKPOINT, index=False)
        done = i + 1
        h = checkpoint_df.at[i, "home_sentiment"]
        print(f"  {done}/{len(recent)}  {row['HomeTeam']} vs {row['AwayTeam']}  → home: {h}")

    time.sleep(0.5)

# Final save & merge

checkpoint_df.to_csv(CHECKPOINT, index=False)

data = data.merge(
    checkpoint_df[["Date", "HomeTeam", "AwayTeam", "home_sentiment", "away_sentiment"]],
    on=["Date", "HomeTeam", "AwayTeam"],
    how="left"
)

# Summary

total     = len(data)
with_sent = data["home_sentiment"].notna().sum()
print(f"\n{'─'*45}")
print(f"Total matches     : {total:,}")
print(f"With sentiment    : {with_sent:,}  ({with_sent/total*100:.1f}%)")
print(f"\nSample scores:")
print(data[data["home_sentiment"].notna()][
    ["Date","HomeTeam","AwayTeam","home_sentiment","away_sentiment"]
].head(10).to_string())

data.to_csv("../Data/combined_data.csv", index=False)
if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)
print("Done!")

Resuming from checkpoint — 2390 already done
  751/2390  Chelsea vs Liverpool  → home: 0.3103
  801/2390  Liverpool vs Norwich  → home: 0.4236
  851/2390  Burnley vs Man City  → home: 0.5057
  901/2390  West Ham vs Arsenal  → home: -0.2394
  951/2390  Man United vs Brighton  → home: -0.1373
  1001/2390  Man United vs Arsenal  → home: -0.1454
  1051/2390  Chelsea vs Man United  → home: 0.3347
  1101/2390  Brighton vs Arsenal  → home: 0.3071
  1151/2390  Man United vs Leeds  → home: 0.1026
  1201/2390  West Ham vs Aston Villa  → home: -0.3806
  1251/2390  Crystal Palace vs Everton  → home: -0.1387
  1301/2390  Bournemouth vs Man United  → home: 0.0028
  1351/2390  Luton vs West Ham  → home: -0.1187
  1401/2390  West Ham vs Newcastle  → home: -0.1197
  1451/2390  Fulham vs Wolves  → home: 0.0669
  1501/2390  Bournemouth vs Fulham  → home: nan
  1551/2390  Wolves vs Brentford  → home: nan
  1601/2390  Burnley vs Brentford  → home: nan
  1651/2390  Fulham vs Liverpool  → home: nan


KeyboardInterrupt: 

In [ ]:
import requests

r = requests.get(
    "https://content.guardianapis.com/search",
    params={
        "q"       : "Arsenal football",
        "section" : "football",
        "page-size": 1,
        "api-key" : GUARDIAN_API_KEY,
    }
)

response = r.json().get("response", {})
status   = response.get("status")
pages    = response.get("pages")

if status == "ok":
    print("API is working — quota refreshed, good to go!")
    print(f"   Test query returned {pages} pages of results")
else:
    print("API not working yet")
    print(f"   Response: {r.json()}")

❌ API not working yet
   Response: {'message': 'API rate limit exceeded'}
